# 第二阶段：基线模型训练
训练原始 YOLOv11n，记录 mAP / FPS 作为对照组

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data='configs/visdrone.yaml',
    epochs=100,
    imgsz=640,  
    batch=-1,  # 自动调整 batch 大小
    workers=8,
    name='baseline_v1',
    project='results',
    plots=True,
    device=0,
)

In [ ]:
# 评估基线
model = YOLO('results/baseline/weights/best.pt')
metrics = model.val(data='configs/visdrone.yaml', imgsz=640)
print(f'mAP@0.5:     {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')

In [ ]:
# 可视化训练曲线
from IPython.display import Image as IPImage
IPImage('results/baseline/results.png')

In [ ]:
# 测试推理速度（FPS）
import time, cv2, numpy as np

dummy = np.zeros((640, 640, 3), dtype=np.uint8)
model = YOLO('results/baseline/weights/best.pt')
# 预热
for _ in range(5):
    model(dummy, verbose=False)
# 计时
t = time.time()
N = 50
for _ in range(N):
    model(dummy, verbose=False)
fps = N / (time.time() - t)
print(f'FPS: {fps:.1f}')